
# 🛍️ Alura Store — Análise de Desempenho de Lojas

### Data Analytics aplicado à tomada de decisão de negócio

Este notebook analisa o desempenho de quatro lojas da **Alura Store** a partir de dados de vendas, categorias, avaliações de clientes, produtos e custos de frete.

O objetivo é responder a uma pergunta de negócio:

> **Qual loja apresenta o menor desempenho e deveria ser considerada prioritariamente para venda?**

A análise foi estruturada para ir além da geração de gráficos, conectando:

**Dados → KPIs → Comparação → Insights → Recomendação → Decisão**

> Projeto desenvolvido no contexto do programa **Oracle Next Education (ONE) / Alura** e reorganizado em formato de portfólio analítico.



## 1. Objetivo da análise

A decisão de vender uma unidade não deve ser baseada em um único indicador.

Por isso, serão avaliadas as quatro lojas considerando:

- faturamento total;
- quantidade de vendas;
- distribuição das vendas por categoria;
- avaliação média dos clientes;
- produtos mais e menos vendidos;
- custo médio de frete.

Ao final, os indicadores serão consolidados para sustentar uma recomendação de negócio com as devidas limitações analíticas.


## 2. Importação das bibliotecas

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 3. Carregamento dos dados

In [ ]:

urls = {
    "Loja 1": "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science/refs/heads/main/base-de-dados-challenge-1/loja_1.csv",
    "Loja 2": "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science/refs/heads/main/base-de-dados-challenge-1/loja_2.csv",
    "Loja 3": "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science/refs/heads/main/base-de-dados-challenge-1/loja_3.csv",
    "Loja 4": "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science/refs/heads/main/base-de-dados-challenge-1/loja_4.csv",
}

lojas = {nome: pd.read_csv(url) for nome, url in urls.items()}

for nome, df in lojas.items():
    print(f"{nome}: {df.shape[0]:,} registros | {df.shape[1]} colunas")


### 3.1 Estrutura dos dados

In [ ]:

lojas["Loja 1"].head()


## 4. Verificação da qualidade dos dados

In [ ]:

qualidade = []

for nome, df in lojas.items():
    qualidade.append({
        "Loja": nome,
        "Registros": len(df),
        "Duplicados": int(df.duplicated().sum()),
        "Valores nulos": int(df.isna().sum().sum())
    })

qualidade_df = pd.DataFrame(qualidade)
qualidade_df



A verificação acima ajuda a identificar problemas básicos de qualidade antes da análise.

Em um projeto real, esta etapa seria complementada por validações de tipos, consistência de datas, regras de negócio e investigação de outliers.


## 5. Consolidação dos dados

In [ ]:

df_completo = pd.concat(
    [df.assign(Loja=nome) for nome, df in lojas.items()],
    ignore_index=True
)

df_completo["Data da Compra"] = pd.to_datetime(
    df_completo["Data da Compra"],
    dayfirst=True,
    errors="coerce"
)

print(f"Base consolidada: {len(df_completo):,} registros")
df_completo.head()


## 6. KPIs principais por loja

In [ ]:

resumo = (
    df_completo
    .groupby("Loja", as_index=False)
    .agg(
        Faturamento=("Preço", "sum"),
        Vendas=("Produto", "count"),
        Avaliacao_Media=("Avaliação da compra", "mean"),
        Frete_Medio=("Frete", "mean")
    )
)

resumo["Ticket_Medio"] = resumo["Faturamento"] / resumo["Vendas"]

resumo = resumo[
    ["Loja", "Faturamento", "Vendas", "Ticket_Medio", "Avaliacao_Media", "Frete_Medio"]
].sort_values("Faturamento", ascending=False)

resumo


### 6.1 Faturamento por loja

In [ ]:

faturamento = resumo.sort_values("Faturamento")

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(faturamento["Loja"], faturamento["Faturamento"])
ax.set_title("Faturamento Total por Loja")
ax.set_xlabel("Faturamento (R$)")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"R$ {x/1_000_000:.2f} mi"))

for i, valor in enumerate(faturamento["Faturamento"]):
    ax.text(valor, i, f"  R$ {valor:,.0f}", va="center")

plt.tight_layout()
plt.show()



**Leitura de negócio:** o faturamento é um dos principais sinais de escala comercial. Porém, ele não deve ser usado isoladamente para determinar a unidade de pior desempenho.


### 6.2 Avaliação média dos clientes

In [ ]:

avaliacoes = resumo.sort_values("Avaliacao_Media")

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(avaliacoes["Loja"], avaliacoes["Avaliacao_Media"])
ax.set_title("Avaliação Média por Loja")
ax.set_xlabel("Nota média")
ax.set_ylabel("")
ax.set_xlim(0, 5)

for i, valor in enumerate(avaliacoes["Avaliacao_Media"]):
    ax.text(valor, i, f"  {valor:.2f}", va="center")

plt.tight_layout()
plt.show()



**Leitura de negócio:** as avaliações são relativamente próximas entre as quatro lojas. Isso indica que satisfação do cliente, sozinha, não explica as diferenças de faturamento.


### 6.3 Frete médio

In [ ]:

fretes = resumo.sort_values("Frete_Medio", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(fretes["Loja"], fretes["Frete_Medio"])
ax.set_title("Frete Médio por Loja")
ax.set_xlabel("Frete médio (R$)")
ax.set_ylabel("")

for i, valor in enumerate(fretes["Frete_Medio"]):
    ax.text(valor, i, f"  R$ {valor:.2f}", va="center")

plt.tight_layout()
plt.show()



**Leitura de negócio:** custos de frete menores podem representar uma vantagem operacional ou comercial. Portanto, uma loja com menor faturamento pode ainda apresentar eficiência logística relevante.


## 7. Vendas por categoria

In [ ]:

categorias = pd.crosstab(
    df_completo["Loja"],
    df_completo["Categoria do Produto"]
)

categorias


In [ ]:

categorias_t = categorias.T

fig, ax = plt.subplots(figsize=(11, 7))
categorias_t.plot(kind="bar", ax=ax)
ax.set_title("Vendas por Categoria e Loja")
ax.set_xlabel("Categoria")
ax.set_ylabel("Quantidade de vendas")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()



A comparação mostra como o mix de vendas se distribui entre as categorias.

Mais importante do que contar quantas categorias existem é analisar **quanto cada categoria representa nas vendas de cada loja**, pois as quatro unidades operam com um portfólio amplo e semelhante.


## 8. Produtos mais e menos vendidos

In [ ]:

for nome, df in lojas.items():
    ranking = df["Produto"].value_counts()

    print(f"\n{nome}")
    print("-" * 40)
    print("Top 5 produtos mais vendidos:")
    display(ranking.head(5).to_frame("Vendas"))

    print("Top 5 produtos menos vendidos:")
    display(ranking.tail(5).sort_values().to_frame("Vendas"))



Em vez de selecionar apenas um produto por loja — o que pode ser afetado por empates — o ranking acima apresenta os cinco produtos de maior e menor volume, oferecendo uma leitura mais robusta do mix.


## 9. Visão comparativa consolidada

In [ ]:

comparativo = resumo.copy()

comparativo["Faturamento"] = comparativo["Faturamento"].map(lambda x: f"R$ {x:,.2f}")
comparativo["Ticket_Medio"] = comparativo["Ticket_Medio"].map(lambda x: f"R$ {x:,.2f}")
comparativo["Avaliacao_Media"] = comparativo["Avaliacao_Media"].map(lambda x: f"{x:.2f}")
comparativo["Frete_Medio"] = comparativo["Frete_Medio"].map(lambda x: f"R$ {x:.2f}")

comparativo


## 10. Rankings dos indicadores

In [ ]:

ranking = resumo.copy()

ranking["Rank_Faturamento"] = ranking["Faturamento"].rank(ascending=False, method="min").astype(int)
ranking["Rank_Avaliacao"] = ranking["Avaliacao_Media"].rank(ascending=False, method="min").astype(int)
ranking["Rank_Frete"] = ranking["Frete_Medio"].rank(ascending=True, method="min").astype(int)

ranking[
    [
        "Loja",
        "Rank_Faturamento",
        "Rank_Avaliacao",
        "Rank_Frete"
    ]
].sort_values("Rank_Faturamento")



Os rankings acima não são combinados em um "score" único porque isso exigiria definir pesos subjetivos entre faturamento, satisfação e frete.

Em vez disso, a decisão é interpretada com transparência, destacando os trade-offs entre os indicadores.


## 11. Principais insights

In [ ]:

menor_faturamento = resumo.loc[resumo["Faturamento"].idxmin()]
maior_faturamento = resumo.loc[resumo["Faturamento"].idxmax()]
melhor_avaliacao = resumo.loc[resumo["Avaliacao_Media"].idxmax()]
menor_avaliacao = resumo.loc[resumo["Avaliacao_Media"].idxmin()]
menor_frete = resumo.loc[resumo["Frete_Medio"].idxmin()]
maior_frete = resumo.loc[resumo["Frete_Medio"].idxmax()]

print(f"Maior faturamento: {maior_faturamento['Loja']} — R$ {maior_faturamento['Faturamento']:,.2f}")
print(f"Menor faturamento: {menor_faturamento['Loja']} — R$ {menor_faturamento['Faturamento']:,.2f}")
print(f"Melhor avaliação: {melhor_avaliacao['Loja']} — {melhor_avaliacao['Avaliacao_Media']:.2f}")
print(f"Menor avaliação: {menor_avaliacao['Loja']} — {menor_avaliacao['Avaliacao_Media']:.2f}")
print(f"Menor frete médio: {menor_frete['Loja']} — R$ {menor_frete['Frete_Medio']:.2f}")
print(f"Maior frete médio: {maior_frete['Loja']} — R$ {maior_frete['Frete_Medio']:.2f}")



### Interpretação

A análise mostra que a decisão não é trivial:

- a loja com menor faturamento não necessariamente apresenta a pior avaliação;
- o menor faturamento pode coexistir com uma vantagem logística;
- as avaliações são relativamente próximas, reduzindo seu poder de discriminação;
- o mix de categorias é amplo nas quatro lojas.

Isso reforça a importância de analisar os indicadores conjuntamente e evitar conclusões baseadas em uma única métrica.


## 12. Recomendação final

In [ ]:

loja_candidata = menor_faturamento["Loja"]
receita_candidata = menor_faturamento["Faturamento"]
avaliacao_candidata = menor_faturamento["Avaliacao_Media"]
frete_candidato = menor_faturamento["Frete_Medio"]

print(
    f"A principal candidata à venda é a {loja_candidata}, "
    f"por apresentar o menor faturamento total (R$ {receita_candidata:,.2f})."
)
print(
    f"Entretanto, sua avaliação média é {avaliacao_candidata:.2f} "
    f"e seu frete médio é R$ {frete_candidato:.2f}, "
    "o que mostra que a unidade não apresenta o pior resultado em todas as dimensões."
)



### Conclusão executiva

Com base nos dados disponíveis, a **Loja 4** é a principal candidata à venda por apresentar o **menor faturamento entre as quatro unidades**.

Entretanto, a recomendação não decorre de uma deterioração generalizada dos indicadores. A Loja 4 apresenta **o menor frete médio** e uma avaliação de clientes próxima às demais unidades.

Portanto, a recomendação é sustentada principalmente por sua **menor capacidade de geração de receita**.

Em um cenário real, a decisão final deveria ser complementada por informações que não estão presentes no dataset, como:

- margem de contribuição;
- custos fixos e operacionais;
- lucro por loja;
- CAC e recorrência de clientes;
- estoque;
- despesas logísticas completas;
- potencial de crescimento regional.

Essa limitação é importante: **Data Analytics deve apoiar a decisão sem extrapolar o que os dados realmente permitem concluir.**


## 13. Competências demonstradas


- Python
- Pandas
- Matplotlib
- Data Cleaning
- Data Wrangling
- Análise Exploratória de Dados (EDA)
- Cálculo de KPIs
- Data Visualization
- Business Analytics
- Análise comparativa
- Geração de insights
- Comunicação analítica
- Tomada de decisão orientada por dados



---

## Autor

**Marcus Guedes**

Marketing | Data Science | Inteligência Artificial | Gestão de Projetos

**GitHub:** MCLG1661

---

> **Transformando dados em informação, informação em insights e insights em decisões.**
